<a href="https://colab.research.google.com/github/kameda-yoshinari/IMISToolExeA/blob/main/800/801_miniGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8-1. Backpropagation by numpy (hand coding)


---
This course material was made with the help / advise of Claude (opus).

---


### What you will learn
By the end of this notebook you will be able to say, from first principles:

1. *"Learning"* a neural network means **minimizing a loss by gradient descent**.
2. **Backpropagation is nothing but the chain rule**, applied mechanically layer by layer.
3. You can **verify** your own gradients with a numerical **gradient check** — no framework, no magic.

You will build a 2-layer classifier for handwritten digits **entirely in NumPy**, writing both the
forward pass and the backward pass by hand. Next week you will see PyTorch's `autograd` reproduce
*exactly* the gradients you write today, and then grow this into a mini language model (a small GPT).

### How this week is graded
- **Task cells** marked 🔧 are your work. Fill them in.
- Your gradients must pass the **gradient check** (relative error `< 1e-6`).
- Your model must reach **test accuracy > 0.95**.
- Short **written answers** at the end are graded on understanding.

### Slots
- **Slot 1:** setup + the forward pass (Tasks 1.1–1.2).
- **Slot 2:** the backward pass + gradient check + training (Tasks 1.3–1.5).
- Watch the recommended backprop video *before* class; use class time to implement and ask questions.

> Runs in seconds on a free CPU (no GPU needed). Works on Google Colab or any local Python.

## Preparation at Google Colab

All the files will be placed on your Google Drive.

In [ ]:
!echo "Start mounting your Google Drive."
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!echo "Make a working folder and move to there."
%cd /content/drive/My\ Drive/
%mkdir -p IMIS_Tool-A/Work800
%cd       IMIS_Tool-A/Work800
!ls

## 0. Setup

We use scikit-learn's built-in `digits` dataset (8×8 grayscale, 10 classes, ~1800 images).
It ships **with** scikit-learn, so there is **no download** — this always works offline.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)

X, y = load_digits(return_X_y=True)   # X: (1797, 64) pixels in [0,16], y: (1797,) labels 0..9
X = X / 16.0                          # normalize pixels to [0, 1]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0)

N, D = Xtr.shape        # N training examples, D=64 input features
C = 10                  # number of classes
H = 64                  # hidden layer width
print(f"train={Xtr.shape}, test={Xte.shape}, classes={C}")

# Peek at a few digits
fig, ax = plt.subplots(1, 8, figsize=(9, 1.4))
for i in range(8):
    ax[i].imshow(Xtr[i].reshape(8, 8), cmap="gray"); ax[i].axis("off"); ax[i].set_title(int(ytr[i]))
plt.tight_layout(); plt.show()

## 1. The model

A 2-layer multilayer perceptron (MLP):

$$
\mathbf{z}_1 = \mathbf{x}\,W_1 + \mathbf{b}_1,\qquad
\mathbf{a}_1 = \mathrm{ReLU}(\mathbf{z}_1),\qquad
\mathbf{z}_2 = \mathbf{a}_1\,W_2 + \mathbf{b}_2
$$

The scores $\mathbf{z}_2$ are turned into probabilities by **softmax**, and we measure error with the
**cross-entropy** loss. For one example with true class $t$:

$$
p_k = \frac{e^{z_{2,k}}}{\sum_j e^{z_{2,j}}},\qquad
\mathcal{L} = -\log p_t
$$

We initialize the weights with He initialization (good for ReLU).

In [ ]:
def init_params():
    return {
        "W1": rng.normal(0, np.sqrt(2/D), (D, H)), "b1": np.zeros(H),
        "W2": rng.normal(0, np.sqrt(2/H), (H, C)), "b2": np.zeros(C),
    }

### 🔧 Task 1.1 & 1.2 — Forward pass

Complete `forward(p, X, y)`. It must return `(loss, cache)` where `loss` is the **mean**
cross-entropy over the batch and `cache` stores whatever the backward pass will need.

Two things to implement:
- **Task 1.1:** the two affine layers and the ReLU.
- **Task 1.2:** the (numerically stable) softmax and the mean cross-entropy loss.

*Hint for stability:* subtract `z2.max(axis=1, keepdims=True)` before `exp`.

In [ ]:
# Quiz version
def forward(p, X, y):
    n = X.shape[0]
    # ┏━━ Task 1.1: affine -> ReLU -> affine ━━
    # >>> SOLUTION
    z1 = 0 # needed
    a1 = 0 # needed
    z2 = 0
    # <<< SOLUTION
    # ┏━━ Task 1.2: stable softmax + mean cross-entropy ━━
    # >>> SOLUTION
    z2s   = 0
    ex    = 0
    probs = 0 # needed
    loss  = 0 # needed
    # <<< SOLUTION
    cache = (X, z1, a1, probs, y)
    return loss, cache

**Sanity check.** Before training, the network is random, so every class is roughly equally likely
($p_k \approx 1/10$) and the loss should be close to $-\log(1/10) = \ln 10 \approx 2.303$.
If you see that number, your forward pass is probably correct.

In [ ]:
p = init_params()
loss0, _ = forward(p, Xtr[:256], ytr[:256])
print(f"initial loss = {loss0:.4f}   (expected ~ {np.log(10):.4f})")

## 2. The backward pass — the heart of the week

We need $\partial \mathcal{L}/\partial W_1, \partial \mathcal{L}/\partial b_1, \partial \mathcal{L}/\partial W_2, \partial \mathcal{L}/\partial b_2$.
Backprop applies the chain rule from the loss backwards. The one fact worth memorizing:

$$
\boxed{\;\frac{\partial \mathcal{L}}{\partial \mathbf{z}_2} = \mathbf{p} - \mathbf{y}_{\text{one-hot}}\;}
$$

Softmax **and** cross-entropy collapse into this beautifully simple form. From there:

$$
dW_2 = \mathbf{a}_1^\top\, d\mathbf{z}_2,\quad
d\mathbf{a}_1 = d\mathbf{z}_2\, W_2^\top,\quad
d\mathbf{z}_1 = d\mathbf{a}_1 \odot \mathbb{1}[\mathbf{z}_1>0],\quad
dW_1 = \mathbf{x}^\top\, d\mathbf{z}_1
$$

(divide the batch gradient by $n$ because the loss is a **mean**).

### 🔧 Task 1.3 — Backward pass

Complete `backward(p, cache)` to return a dict of gradients with the same keys as `p`
(`"W1","b1","W2","b2"`). Follow the equations above. The bias gradients are the column sums
of the corresponding `dz`.

In [ ]:
# Quiz version
def backward(p, cache):
    X, z1, a1, probs, y = cache
    n = X.shape[0]
    # ┏━━ Task 1.3: chain rule, output layer -> hidden layer ━━
    # >>> SOLUTION
    dz2 = 0
    dz2[np.arange(n), y] = 0
    dz2 = 0                             # loss is a mean
    dW2 = 0 # needed
    db2 = 0 # needed
    da1 = 0
    dz1 = 0                             # ReLU derivative
    dW1 = 0 # needed
    db1 = 0 # needed
    # <<< SOLUTION
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}

### Gradient check (provided) — verify your own math

We compare your analytical gradient against a **numerical** estimate
$\frac{\mathcal{L}(\theta+\epsilon) - \mathcal{L}(\theta-\epsilon)}{2\epsilon}$ for a sample of
parameters. If your `backward` is correct, the worst relative error will be tiny (around `1e-8`).

**Your Task 1.3 is accepted when the printed error is `< 1e-6`.**

In [ ]:
def gradient_check(seed=0):
    p = init_params()
    Xb, yb = Xtr[:16], ytr[:16]
    _, cache = forward(p, Xb, yb)
    grads = backward(p, cache)
    eps, worst = 1e-5, 0.0
    gen = np.random.default_rng(seed)
    for k in ["W1", "b1", "W2", "b2"]:
        for _ in range(25):
            idx = tuple(gen.integers(0, s) for s in p[k].shape)
            old = p[k][idx]
            p[k][idx] = old + eps; lp, _ = forward(p, Xb, yb)
            p[k][idx] = old - eps; lm, _ = forward(p, Xb, yb)
            p[k][idx] = old
            num = (lp - lm) / (2 * eps); ana = grads[k][idx]
            worst = max(worst, abs(num - ana) / max(1e-8, abs(num) + abs(ana)))
    return worst

err = gradient_check()
print(f"worst relative error = {err:.2e}   ->  {'PASS ✅' if err < 1e-6 else 'FAIL ❌'}")

## 3. Train it

Now that the gradients are trustworthy, training is just: repeatedly compute gradients on a
mini-batch and step the parameters downhill. Watch the loss fall and the test accuracy rise.

In [ ]:
def accuracy(p, X, y):
    _, cache = forward(p, X, y)
    return (cache[3].argmax(axis=1) == y).mean()

def train(lr=0.5, epochs=50, batch_size=64, seed=0):
    p = init_params()
    gen = np.random.default_rng(seed)
    history = []
    for epoch in range(epochs):
        for i in range(0, N, batch_size):
            b = gen.permutation(N)[i:i+batch_size]
            loss, cache = forward(p, Xtr[b], ytr[b])
            g = backward(p, cache)
            for k in p:
                p[k] -= lr * g[k]
        acc = accuracy(p, Xte, yte)
        history.append((epoch, loss, acc))
    return p, history

p, hist = train()
ep, ls, ac = zip(*hist)
print(f"final test accuracy = {ac[-1]:.4f}   ->  {'PASS ✅' if ac[-1] > 0.95 else 'FAIL ❌'}")

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(ep, ls); ax[0].set_title("training loss"); ax[0].set_xlabel("epoch")
ax[1].plot(ep, ac); ax[1].set_title("test accuracy"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

### 🔧 Task 1.4 — Experiment

Change **one** hyperparameter (hidden width `H`, learning rate `lr`, or number of `epochs`) and
observe the effect on the final test accuracy. Keep the change in the cell below and note what you saw.

*Tip:* to change `H`, set `H = ...` and re-run the `init_params`, forward/backward, and training cells,
or copy a minimal training run into the cell below.

In [ ]:
# Quiz version
# ┏━━ Task 1.4: run one experiment and print the result ━━
# >>> SOLUTION
p2, hist2 = 0
print(f"lr=0.1  -> final test accuracy = {hist2[-1][2]:.4f}")
# <<< SOLUTION

## Assignment 8-1 — submit this notebook

**Automatic checks (must pass):**
1. `gradient_check()` prints a relative error `< 1e-6`.
2. The trained model reaches test accuracy > 0.95.

**Written answers (🔧 Task 1.5).** Answer in the markdown cells below, 2–4 sentences each.

**A.** Why does the gradient of softmax + cross-entropy simplify to $\mathbf{p}-\mathbf{y}_{\text{one-hot}}$?
What does each term mean intuitively?

**B.** In `backward`, why do we multiply the gradient by `(z1 > 0)` for the ReLU layer?
What happens to the gradient of a "dead" unit whose input was negative?

**C.** From your Task 1.4 experiment: what changed, and why do you think it behaved that way?

> Next week we let PyTorch's `autograd` compute these very gradients automatically — and we will
> confirm they match the ones you wrote today — before growing this into a mini-GPT.

**A.** *(your answer here)*

---

**B.** *(your answer here)*

---

**C.** *(your answer here)*

---
---

**X.** *(your student ID)*

---

**Y.** *(your name)*



---
Tools and Practices for Intelligent Interaction Systems A  
Master's and Docotal programs in intelligent and mechanical interaction systems, University of Tsukuba, Japan.  
KAMEDA Yoshinari, SHIBUYA Takeshi  

知能システムツール演習a  
知能機能システム学位プログラム (筑波大学大学院)  
担当：亀田能成，澁谷長史  

2026/07/27.  
